# Notebook 10 — Advanced Feature Engineering

**Goal:** Extend the 29-feature baseline with 15 new features across 6 tiers. Run a full 5-fold LGBM CV after each tier to measure marginal gain. Drop any tier adding < +0.001 CV AUC. Target: CV AUC ≥ 0.875.

**Inputs:** `data/raw/train.csv`, `data/raw/test.csv`, `data/processed/fold_assignments.parquet`  
**Outputs:** `data/processed/train_features_v2.parquet`, `data/processed/test_features_v2.parquet`

**Self-contained:** all helpers defined inline — no local imports. Runs on Kaggle without modification.

| Tier | Features | Key signal |
|------|---------|------------|
| 1 | Compound-Weighted Age (3) | `TyreLife×ordinal` corr=0.298 vs 0.274 for TyreLife alone |
| 2 | Pit Window Signal (2) | 37.3% pit rate in RaceProgress 40–70% vs 14.2% outside (2.6×) |
| 3 | Position Volatility (2) | Erratic position swings = degraded tyre signal |
| 4 | Stint Completion (2) | `laps_to_driver_end` corr=−0.236 vs −0.185 for `laps_remaining` |
| 5 | Cross-Driver Field (3) | Field median laptime as SC/VSC proxy; relative pace gap |
| 6 | Enhanced Target Encodings (3) | Race×Year (104 groups), Driver×Compound rate, TyreLife vs typical |

In [1]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')

# Robust project root detection — works from workspace root, notebooks/, or Kaggle
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root  = cwd
data_dir      = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
processed_dir.mkdir(parents=True, exist_ok=True)

# Kaggle path fallback
if not data_dir.exists():
    data_dir = Path('/kaggle/input/playground-series-s6e5')
    processed_dir = Path('/kaggle/working')

assert data_dir.exists(), f'Data directory not found: {data_dir}'
print(f'Project root : {project_root}')
print(f'Data dir     : {data_dir}')
print(f'Processed dir: {processed_dir}')

Project root : c:\Repos\predict-f1-pit-stops
Data dir     : c:\Repos\predict-f1-pit-stops\data\raw
Processed dir: c:\Repos\predict-f1-pit-stops\data\processed


In [2]:
train_raw = pd.read_csv(data_dir / 'train.csv')
test_raw  = pd.read_csv(data_dir / 'test.csv')
folds     = pd.read_parquet(processed_dir / 'fold_assignments.parquet')

print(f'Train: {train_raw.shape}  |  Test: {test_raw.shape}')
print(f'Fold counts:\n{folds["fold"].value_counts().sort_index()}')
print(f'\nTrain columns: {list(train_raw.columns)}')

Train: (439140, 16)  |  Test: (188165, 15)
Fold counts:
fold
0    88018
1    87444
2    88027
3    87854
4    87797
Name: count, dtype: int64

Train columns: ['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']


## Tier 5 — Cross-Driver Field Features (compute first on combined dataset)

Field-level features (median laptime across all drivers per lap) must be computed from the **combined** train+test dataset **before** any CV split. Reason: the median is purely contemporaneous — it aggregates over all drivers on the same lap, using only `LapTime (s)` which is present in both train and test. There is no target leakage because `PitNextLap` is never used in the computation.

Computing on combined gives better estimates for laps where few drivers are recorded (e.g., retirements reduce field size). Merge back by `(Race, Year, LapNumber)` after.

In [3]:
# Combine train and test on common columns (no PitNextLap in test)
common_cols = [c for c in train_raw.columns if c != 'PitNextLap']
combined    = pd.concat([train_raw[common_cols], test_raw[common_cols]], ignore_index=True)

# Per-lap field median and lap-over-lap pace change
field = (
    combined
    .groupby(['Race', 'Year', 'LapNumber'])['LapTime (s)']
    .median()
    .reset_index()
    .rename(columns={'LapTime (s)': 'field_median_laptime'})
)
field = field.sort_values(['Race', 'Year', 'LapNumber'])
field['field_pace_change'] = (
    field.groupby(['Race', 'Year'])['field_median_laptime']
    .diff(1)
    .fillna(0.0)
)

print(f'Field feature table: {field.shape}')
print(field.head(5).to_string(index=False))

# Verify field_pace_change spikes correlate with SC/VSC events
# (safety car = all drivers slow suddenly = large positive field_pace_change)
top_spikes = field.nlargest(5, 'field_pace_change')[['Race', 'Year', 'LapNumber', 'field_pace_change']]
print('\nTop field_pace_change spikes (likely SC/VSC laps):')
print(top_spikes.to_string(index=False))

Field feature table: (6316, 5)
                Race  Year  LapNumber  field_median_laptime  field_pace_change
Abu Dhabi Grand Prix  2022          1                98.621              0.000
Abu Dhabi Grand Prix  2022          2                94.791             -3.830
Abu Dhabi Grand Prix  2022          3               100.689              5.898
Abu Dhabi Grand Prix  2022          4                90.433            -10.256
Abu Dhabi Grand Prix  2022          5                91.223              0.790

Top field_pace_change spikes (likely SC/VSC laps):
                     Race  Year  LapNumber  field_pace_change
         Qatar Grand Prix  2024         58            60.6870
         Qatar Grand Prix  2022         19            54.7675
Emilia Romagna Grand Prix  2023         44            53.5000
       British Grand Prix  2023         37            44.1340
       Pre-Season Testing  2023         37            44.0035


## `build_base_features_v2()` — All Tiers 1–4 Included

This function extends NB 02's `build_base_features()` with 4 new tiers of features (10 new columns). Tier 5 (field features) is merged separately after this call because it requires the combined train+test dataset. Tier 6 (enhanced target encodings) is applied fold-aware in `apply_target_encodings_v2()`.

Key groupby rules preserved:
- **Lag/rolling** features: `(Race, Year, Driver, Stint)` — prevents cross-stint and cross-driver contamination
- **Position volatility** (`pos_change_rolling_std_3`): `(Race, Year, Driver)` only — intentionally spans stint boundaries to carry late-stint signal

In [4]:
def build_base_features_v2(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build all non-target-encoded features (26 base from NB 02 + 10 new from Tiers 1-4).
    Tier 5 (field features) must be merged separately.
    Tier 6 (enhanced target encodings) is applied inside CV folds via apply_target_encodings_v2().
    Sorts by (Race, Year, LapNumber) internally.
    """
    CLIFF       = {'SOFT': 13, 'MEDIUM': 49, 'HARD': 61, 'INTERMEDIATE': 1, 'WET': 1}
    ORDINAL     = {'SOFT':  1, 'MEDIUM':  2, 'HARD':  3, 'INTERMEDIATE': 0, 'WET': 0}
    W_LO, W_HI  = -205, 122
    STINT_KEYS   = ['Race', 'Year', 'Driver', 'Stint']
    DRIVER_KEYS  = ['Race', 'Year', 'Driver']

    df = df.copy().sort_values(['Race', 'Year', 'LapNumber']).reset_index(drop=True)

    # ── Original 26 base features (NB 02) ─────────────────────────────────────
    df['TyreLife_normalized_by_compound'] = df['TyreLife'] / df['Compound'].map(CLIFF)
    df['TyreLife_sq']                     = df['TyreLife'] ** 2
    df['is_wet_tyre']      = df['Compound'].isin({'INTERMEDIATE', 'WET'}).astype(np.int8)
    df['compound_ordinal'] = df['Compound'].map(ORDINAL).astype(np.int8)
    df['laps_remaining']     = 1.0 - df['RaceProgress']
    df['is_testing_session'] = (df['Race'] == 'Pre-Season Testing').astype(np.int8)
    df['Cumulative_Degradation_winsorized'] = df['Cumulative_Degradation'].clip(W_LO, W_HI)
    df['Degradation_rate'] = df['Cumulative_Degradation_winsorized'] / df['TyreLife'].clip(lower=1)
    df['Degradation_acceleration'] = (
        df.groupby(STINT_KEYS)['Cumulative_Degradation'].diff(1).fillna(0.0)
    )
    for raw_col, feat_base in [('LapTime (s)', 'LapTime'), ('LapTime_Delta', 'LapTime_Delta')]:
        grp = df.groupby(STINT_KEYS)[raw_col]
        for lag in [1, 2, 3]:
            df[f'{feat_base}_lag{lag}'] = grp.shift(lag).fillna(0.0)
    for w in [3, 5]:
        df[f'LapTime_rolling_mean_{w}'] = (
            df.groupby(STINT_KEYS)['LapTime (s)']
            .rolling(w, min_periods=1).mean()
            .droplevel(list(range(len(STINT_KEYS)))).sort_index()
        )
        df[f'LapTime_rolling_std_{w}'] = (
            df.groupby(STINT_KEYS)['LapTime (s)']
            .rolling(w, min_periods=2).std()
            .droplevel(list(range(len(STINT_KEYS)))).sort_index().fillna(0.0)
        )
    deg_grp = df.groupby(STINT_KEYS)['Cumulative_Degradation']
    for w in [3, 5]:
        df[f'Degradation_rolling_slope_{w}'] = (
            (df['Cumulative_Degradation'] - deg_grp.shift(w - 1)) / (w - 1)
        ).fillna(0.0)
    df['TyreLife_x_laps_remaining']  = df['TyreLife'] * df['laps_remaining']
    df['Degradation_x_RaceProgress'] = df['Cumulative_Degradation_winsorized'] * df['RaceProgress']
    df['Position_x_RaceProgress']    = df['Position'] * df['RaceProgress']

    # ── Tier 1: Compound-Weighted Age (3 features) ─────────────────────────────
    # compound_ordinal: SOFT=1, MED=2, HARD=3, wet=0. Products weight tyre age by durability.
    # TyreLife_x_compound_ordinal: corr=0.298 vs 0.274 for TyreLife alone
    df['TyreLife_x_compound_ordinal'] = df['TyreLife'] * df['compound_ordinal']
    df['Stint_x_compound_ordinal']    = df['Stint'] * df['compound_ordinal']
    df['TyreLife_x_cmpd_x_laps_rem'] = df['TyreLife'] * df['compound_ordinal'] * df['laps_remaining']

    # ── Tier 2: Pit Window Signal (2 features) ─────────────────────────────────
    # RaceProgress 40–70%: 37.3% pit rate vs 14.2% outside (2.6x). DT finds this in
    # first 3 splits; LGBM with reg_alpha=9.79 must reconstruct it indirectly.
    df['prime_pit_window']       = (
        (df['RaceProgress'] >= 0.4) & (df['RaceProgress'] < 0.7)
    ).astype(np.int8)
    df['prime_window_x_compound'] = df['prime_pit_window'] * df['compound_ordinal']

    # ── Tier 3: Position Volatility (2 features) ───────────────────────────────
    df['abs_position_change'] = df['Position_Change'].abs()
    # Group by Driver only (NOT Stint) — erratic position loss at stint-end is a pit signal
    df['pos_change_rolling_std_3'] = (
        df.groupby(DRIVER_KEYS)['Position_Change']
        .rolling(3, min_periods=1).std()
        .droplevel(list(range(len(DRIVER_KEYS)))).sort_index().fillna(0.0)
    )

    # ── Tier 4: Stint Completion Signals (2 features) ──────────────────────────
    # PitStop_lag1: safe — PitStop is in both train and test. Lagged to avoid temporal leakage.
    df['PitStop_lag1'] = (
        df.groupby(DRIVER_KEYS)['PitStop'].shift(1).fillna(0).astype(np.int8)
    )
    # laps_to_driver_end: corr=-0.236 vs -0.185 for laps_remaining; captures DNF asymmetry
    df['laps_to_driver_end'] = (
        df.groupby(DRIVER_KEYS)['LapNumber'].transform('max') - df['LapNumber']
    )

    return df

print('build_base_features_v2 defined.')

build_base_features_v2 defined.


## `apply_target_encodings_v2()` — Original 3 + 3 New Tier 6 Encodings

All 6 encodings are computed from the **training fold only** — never from the full dataset. Unseen combinations fall back to the global mean (or global median for `TyreLife_vs_driver_typical`).

New encodings:
- **`Race_Year_target_encoded`** — 104 (Race, Year) groups vs 26 Race-only groups. Captures circuit-year-specific strategy shifts (e.g., 2023 vs 2024 regulations).
- **`Driver_Compound_target_encoded`** — pit rate per (Driver, Compound) pair. Captures tyre-strategy tendencies specific to each driver-compound combination.
- **`TyreLife_vs_driver_typical`** — `TyreLife − median(TyreLife at pits for this Driver+Compound)`. Positive = driver has exceeded their historical stop threshold = high pit urgency.

In [5]:
def apply_target_encodings_v2(train_df: pd.DataFrame, val_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute all 6 target encodings from train_df; apply to val_df.
    Call INSIDE each CV fold:
        train_df = fold's training rows (must contain PitNextLap)
        val_df   = fold's validation rows OR the test set
    Never call with the full dataset — that leaks the target.
    """
    global_mean    = train_df['PitNextLap'].mean()
    pit_rows       = train_df[train_df['PitNextLap'] == 1]
    global_typical = pit_rows['TyreLife'].median()

    # --- Pre-compute all encoding maps ---
    race_map        = train_df.groupby('Race')['PitNextLap'].mean()
    driver_map      = train_df.groupby('Driver')['PitNextLap'].mean()
    stint_map       = (
        train_df.groupby(['Driver', 'Race', 'Year', 'Stint'])['TyreLife']
        .max().reset_index().groupby('Driver')['TyreLife'].mean()
    )
    global_stint    = stint_map.mean()

    # New Tier 6 maps (MultiIndex Series → dict for fast lookup)
    race_year_dict   = train_df.groupby(['Race', 'Year'])['PitNextLap'].mean().to_dict()
    driver_cmpd_dict = train_df.groupby(['Driver', 'Compound'])['PitNextLap'].mean().to_dict()
    typical_tl_dict  = pit_rows.groupby(['Driver', 'Compound'])['TyreLife'].median().to_dict()

    out = val_df.copy()

    # --- Original 3 encodings ---
    out['Race_target_encoded']     = out['Race'].map(race_map).fillna(global_mean)
    out['Driver_target_encoded']   = out['Driver'].map(driver_map).fillna(global_mean)
    out['Driver_avg_stint_length'] = out['Driver'].map(stint_map).fillna(global_stint)

    # --- New Tier 6 encodings ---
    out['Race_Year_target_encoded'] = [
        race_year_dict.get((r, y), global_mean)
        for r, y in zip(out['Race'], out['Year'])
    ]
    out['Driver_Compound_target_encoded'] = [
        driver_cmpd_dict.get((d, c), global_mean)
        for d, c in zip(out['Driver'], out['Compound'])
    ]
    typical_tl = np.array([
        typical_tl_dict.get((d, c), global_typical)
        for d, c in zip(out['Driver'], out['Compound'])
    ], dtype=float)
    out['TyreLife_vs_driver_typical'] = out['TyreLife'].values - typical_tl

    return out

print('apply_target_encodings_v2 defined.')

apply_target_encodings_v2 defined.


In [6]:
# Build all features for train and test
t0 = time.time()
train = build_base_features_v2(train_raw)
test  = build_base_features_v2(test_raw)

# Merge Tier 5 field features (computed on combined before CV split)
train = train.merge(field, on=['Race', 'Year', 'LapNumber'], how='left')
test  = test.merge(field,  on=['Race', 'Year', 'LapNumber'], how='left')

# laptime_vs_field — individual driver pace relative to field median
train['laptime_vs_field'] = train['LapTime (s)'] - train['field_median_laptime']
test['laptime_vs_field']  = test['LapTime (s)']  - test['field_median_laptime']

# Merge fold assignments
train = train.merge(folds, on='id', how='left')

elapsed = time.time() - t0
base_cols = train_raw.shape[1]
new_cols  = train.shape[1] - base_cols - 1  # -1 for fold column
print(f'Done in {elapsed:.1f}s')
print(f'Train: {train_raw.shape} → {train.shape}  ({new_cols} new feature columns + fold)')
print(f'Test : {test_raw.shape}  → {test.shape}')
print(f'Fold col present: {"fold" in train.columns}')

Done in 45.6s
Train: (439140, 16) → (439140, 53)  (36 new feature columns + fold)
Test : (188165, 15)  → (188165, 51)
Fold col present: True


In [7]:
# Verify new feature signal vs PitNextLap
NEW_FEATURES = [
    # Tier 1
    'TyreLife_x_compound_ordinal', 'Stint_x_compound_ordinal', 'TyreLife_x_cmpd_x_laps_rem',
    # Tier 2
    'prime_pit_window', 'prime_window_x_compound',
    # Tier 3
    'abs_position_change', 'pos_change_rolling_std_3',
    # Tier 4
    'PitStop_lag1', 'laps_to_driver_end',
    # Tier 5
    'field_median_laptime', 'laptime_vs_field', 'field_pace_change',
]

print('New feature correlations with PitNextLap:')
corrs = train[NEW_FEATURES].corrwith(train['PitNextLap']).abs().sort_values(ascending=False)
for feat, corr in corrs.items():
    print(f'  {feat:<40} {corr:.4f}')

print('\nNull check for new features:')
nulls = train[NEW_FEATURES].isnull().sum()
if nulls.any():
    print('WARNING:')
    print(nulls[nulls > 0])
else:
    print('  PASSED — no nulls')

# Verify prime_pit_window pit rate
print('\nprime_pit_window pit rate validation:')
pit_rates = train.groupby('prime_pit_window')['PitNextLap'].agg(['mean', 'count'])
print(pit_rates.rename(index={0: 'outside window', 1: 'inside window'}))
ratio = pit_rates.loc[1, 'mean'] / pit_rates.loc[0, 'mean']
print(f'  Ratio: {ratio:.2f}× (expected ~2.6×)')

New feature correlations with PitNextLap:
  TyreLife_x_compound_ordinal              0.2977
  TyreLife_x_cmpd_x_laps_rem               0.2779
  prime_window_x_compound                  0.2617
  Stint_x_compound_ordinal                 0.2570
  prime_pit_window                         0.2482
  laps_to_driver_end                       0.2364
  pos_change_rolling_std_3                 0.2035
  abs_position_change                      0.1647
  PitStop_lag1                             0.1312
  field_median_laptime                     0.0541
  field_pace_change                        0.0356
  laptime_vs_field                         0.0067

Null check for new features:
  PASSED — no nulls

prime_pit_window pit rate validation:
                      mean   count
prime_pit_window                  
outside window    0.142477  331358
inside window     0.372697  107782
  Ratio: 2.62× (expected ~2.6×)


## CV Helper + LGBM Params

Using the Optuna-tuned params from Notebook 05 for all CV runs. The key param is `reg_alpha=9.79` (extreme L1 — zeros out weak features). Tier evaluation measures whether each tier's features survive this regularisation.

Early stopping: 50 rounds patience. `apply_target_encodings_v2()` is called inside each fold — the v2 function returns all 6 target encoding columns, and the feature list argument selects which subset to use per tier.

In [8]:
# Optuna-tuned params from Notebook 05 (fold 0, 100 trials)
LGBM_BEST = dict(
    n_estimators      = 2000,
    learning_rate     = 0.049228,
    num_leaves        = 62,
    min_child_samples = 91,
    subsample         = 0.752871,
    colsample_bytree  = 0.746388,
    reg_alpha         = 9.791678,
    reg_lambda        = 0.0,
    random_state      = 42,
    verbose           = -1,
)

def run_lgbm_cv(df, feature_cols, lgbm_params=None, label=''):
    """5-fold GroupKFold CV. df must have 'fold' and 'PitNextLap' columns.
    apply_target_encodings_v2 is called inside each fold.
    Returns (oof_auc, fold_aucs_array, oof_preds_array).
    """
    if lgbm_params is None:
        lgbm_params = LGBM_BEST
    oof  = np.zeros(len(df))
    aucs = []
    t0   = time.time()

    for fold_idx in range(5):
        tr_idx  = np.where(df['fold'] != fold_idx)[0]
        val_idx = np.where(df['fold'] == fold_idx)[0]

        train_sub = apply_target_encodings_v2(df.iloc[tr_idx], df.iloc[tr_idx])
        val_sub   = apply_target_encodings_v2(df.iloc[tr_idx], df.iloc[val_idx])

        missing = [f for f in feature_cols if f not in train_sub.columns]
        if missing:
            raise KeyError(f'Missing features in fold {fold_idx}: {missing}')

        X_tr, y_tr   = train_sub[feature_cols].to_numpy(), train_sub['PitNextLap'].to_numpy()
        X_val, y_val = val_sub[feature_cols].to_numpy(),   val_sub['PitNextLap'].to_numpy()

        callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        m = lgb.LGBMClassifier(**lgbm_params)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=callbacks)

        oof[val_idx] = m.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, oof[val_idx])
        aucs.append(fold_auc)
        print(f'  Fold {fold_idx}: AUC={fold_auc:.4f}  trees={m.best_iteration_}')

    oof_auc = roc_auc_score(df['PitNextLap'], oof)
    elapsed = time.time() - t0
    print(f'  ─── OOF AUC: {oof_auc:.4f}  std={np.std(aucs):.4f}  n_feat={len(feature_cols)}  ({elapsed:.0f}s)')
    if label:
        print(f'  ─── {label}')
    return oof_auc, np.array(aucs), oof

print('CV helper defined. LGBM params:')
for k, v in LGBM_BEST.items():
    print(f'  {k}: {v}')

CV helper defined. LGBM params:
  n_estimators: 2000
  learning_rate: 0.049228
  num_leaves: 62
  min_child_samples: 91
  subsample: 0.752871
  colsample_bytree: 0.746388
  reg_alpha: 9.791678
  reg_lambda: 0.0
  random_state: 42
  verbose: -1


## Baseline CV — Reproduce NB 05 (29 features)

In [9]:
BASE_FEATURES = [
    'TyreLife_normalized_by_compound', 'TyreLife_sq',
    'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session',
    'Stint', 'Position',
    'Cumulative_Degradation_winsorized', 'Degradation_rate', 'Degradation_acceleration',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3',  'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]
TARGET_ENC_V1 = ['Race_target_encoded', 'Driver_target_encoded', 'Driver_avg_stint_length']

FEATS_BASELINE = BASE_FEATURES + TARGET_ENC_V1  # 29 features

print(f'Baseline: {len(FEATS_BASELINE)} features — reproducing NB 05 (expected OOF AUC ~0.8558)')
print('Running 5-fold CV...')
auc_baseline, aucs_baseline, _ = run_lgbm_cv(train, FEATS_BASELINE, label='Baseline (NB 05 params)')

Baseline: 29 features — reproducing NB 05 (expected OOF AUC ~0.8558)
Running 5-fold CV...
  Fold 0: AUC=0.8686  trees=284
  Fold 1: AUC=0.8246  trees=35
  Fold 2: AUC=0.8404  trees=41
  Fold 3: AUC=0.8893  trees=462
  Fold 4: AUC=0.8670  trees=294
  ─── OOF AUC: 0.8558  std=0.0228  n_feat=29  (58s)
  ─── Baseline (NB 05 params)


## Tier 1 — Compound-Weighted Age (+3 features)

In [10]:
TIER1_FEATURES = [
    'TyreLife_x_compound_ordinal',  # corr=0.298 vs 0.274 for TyreLife alone
    'Stint_x_compound_ordinal',
    'TyreLife_x_cmpd_x_laps_rem',
]
FEATS_T1 = FEATS_BASELINE + TIER1_FEATURES  # 32 features

print(f'Tier 1: {len(FEATS_T1)} features (+{len(TIER1_FEATURES)} compound-weighted age)')
print('Running 5-fold CV...')
auc_t1, aucs_t1, _ = run_lgbm_cv(train, FEATS_T1, label='Tier 1: Compound-Weighted Age')
delta_t1 = auc_t1 - auc_baseline
print(f'  Δ vs baseline: {delta_t1:+.4f}  ({"KEEP" if delta_t1 >= 0.001 else "DROP — < 0.001 threshold"})')

Tier 1: 32 features (+3 compound-weighted age)
Running 5-fold CV...
  Fold 0: AUC=0.8652  trees=210
  Fold 1: AUC=0.8241  trees=30
  Fold 2: AUC=0.8376  trees=50
  Fold 3: AUC=0.8887  trees=418
  Fold 4: AUC=0.8662  trees=195
  ─── OOF AUC: 0.8550  std=0.0229  n_feat=32  (54s)
  ─── Tier 1: Compound-Weighted Age
  Δ vs baseline: -0.0009  (DROP — < 0.001 threshold)


## Tier 2 — Pit Window Signal (+2 features)

In [11]:
TIER2_FEATURES = [
    'prime_pit_window',       # binary: RaceProgress 40–70% = 37.3% pit rate vs 14.2% outside
    'prime_window_x_compound',
]
FEATS_T2 = FEATS_T1 + TIER2_FEATURES  # always cumulate; drop decisions in summary table

print(f'Tier 2: {len(FEATS_T2)} features (+{len(TIER2_FEATURES)} pit window signal)')
print('Running 5-fold CV...')
auc_t2, aucs_t2, _ = run_lgbm_cv(train, FEATS_T2, label='Tier 2: Pit Window Signal')
delta_t2 = auc_t2 - auc_t1
print(f'  Δ vs Tier 1: {delta_t2:+.4f}  ({"KEEP" if delta_t2 >= 0.001 else "DROP — < 0.001 threshold"})')

Tier 2: 34 features (+2 pit window signal)
Running 5-fold CV...
  Fold 0: AUC=0.8630  trees=96
  Fold 1: AUC=0.8307  trees=70
  Fold 2: AUC=0.8416  trees=70
  Fold 3: AUC=0.8879  trees=572
  Fold 4: AUC=0.8632  trees=76
  ─── OOF AUC: 0.8586  std=0.0198  n_feat=34  (53s)
  ─── Tier 2: Pit Window Signal
  Δ vs Tier 1: +0.0036  (KEEP)


## Tier 3 — Position Volatility (+2 features)

In [12]:
TIER3_FEATURES = [
    'abs_position_change',
    'pos_change_rolling_std_3',  # grouped by (Race,Year,Driver) only — spans stint boundaries
]
FEATS_T3 = FEATS_T2 + TIER3_FEATURES

print(f'Tier 3: {len(FEATS_T3)} features (+{len(TIER3_FEATURES)} position volatility)')
print('Running 5-fold CV...')
auc_t3, aucs_t3, _ = run_lgbm_cv(train, FEATS_T3, label='Tier 3: Position Volatility')
delta_t3 = auc_t3 - auc_t2
print(f'  Δ vs Tier 2: {delta_t3:+.4f}  ({"KEEP" if delta_t3 >= 0.001 else "DROP — < 0.001 threshold"})')

Tier 3: 36 features (+2 position volatility)
Running 5-fold CV...
  Fold 0: AUC=0.9063  trees=325
  Fold 1: AUC=0.8787  trees=80
  Fold 2: AUC=0.9038  trees=68
  Fold 3: AUC=0.9121  trees=377
  Fold 4: AUC=0.8904  trees=211
  ─── OOF AUC: 0.8977  std=0.0121  n_feat=36  (57s)
  ─── Tier 3: Position Volatility
  Δ vs Tier 2: +0.0391  (KEEP)


## Tier 4 — Stint Completion Signals (+2 features)

In [13]:
TIER4_FEATURES = [
    'PitStop_lag1',       # was previous lap a pit-in? These laps have very low pit prob.
    'laps_to_driver_end', # corr=-0.236 vs -0.185 for laps_remaining; captures DNF asymmetry
]
FEATS_T4 = FEATS_T3 + TIER4_FEATURES

print(f'Tier 4: {len(FEATS_T4)} features (+{len(TIER4_FEATURES)} stint completion)')
print('Running 5-fold CV...')
auc_t4, aucs_t4, _ = run_lgbm_cv(train, FEATS_T4, label='Tier 4: Stint Completion Signals')
delta_t4 = auc_t4 - auc_t3
print(f'  Δ vs Tier 3: {delta_t4:+.4f}  ({"KEEP" if delta_t4 >= 0.001 else "DROP — < 0.001 threshold"})')

Tier 4: 38 features (+2 stint completion)
Running 5-fold CV...
  Fold 0: AUC=0.9089  trees=343
  Fold 1: AUC=0.8759  trees=78
  Fold 2: AUC=0.9036  trees=92
  Fold 3: AUC=0.9151  trees=700
  Fold 4: AUC=0.8915  trees=187
  ─── OOF AUC: 0.8987  std=0.0139  n_feat=38  (63s)
  ─── Tier 4: Stint Completion Signals
  Δ vs Tier 3: +0.0010  (DROP — < 0.001 threshold)


## Tier 5 — Cross-Driver Field Features (+3 features)

In [14]:
TIER5_FEATURES = [
    'field_median_laptime', # median laptime across all drivers on this lap
    'laptime_vs_field',     # this driver's pace vs field median
    'field_pace_change',    # lap-over-lap change in field median — SC/VSC proxy
]
FEATS_T5 = FEATS_T4 + TIER5_FEATURES

print(f'Tier 5: {len(FEATS_T5)} features (+{len(TIER5_FEATURES)} field features)')
print('Running 5-fold CV...')
auc_t5, aucs_t5, _ = run_lgbm_cv(train, FEATS_T5, label='Tier 5: Cross-Driver Field Features')
delta_t5 = auc_t5 - auc_t4
print(f'  Δ vs Tier 4: {delta_t5:+.4f}  ({"KEEP" if delta_t5 >= 0.001 else "DROP — < 0.001 threshold"})')

Tier 5: 41 features (+3 field features)
Running 5-fold CV...
  Fold 0: AUC=0.9043  trees=176
  Fold 1: AUC=0.8769  trees=77
  Fold 2: AUC=0.9049  trees=77
  Fold 3: AUC=0.9119  trees=333
  Fold 4: AUC=0.8948  trees=219
  ─── OOF AUC: 0.8982  std=0.0121  n_feat=41  (56s)
  ─── Tier 5: Cross-Driver Field Features
  Δ vs Tier 4: -0.0005  (DROP — < 0.001 threshold)


## Tier 6 — Enhanced Target Encodings (+3 features)

These are fold-aware — already computed by `apply_target_encodings_v2()`. Adding them to the feature list for the CV run includes them in the model.

In [15]:
TARGET_ENC_V2 = [
    'Race_Year_target_encoded',        # 104 (Race,Year) groups vs 26 Race-only
    'Driver_Compound_target_encoded',  # per-driver compound pit rate
    'TyreLife_vs_driver_typical',      # TyreLife - median(TyreLife at pits for Driver+Compound)
]
FEATS_T6 = FEATS_T5 + TARGET_ENC_V2  # All 6 target encodings now included

print(f'Tier 6: {len(FEATS_T6)} features (+{len(TARGET_ENC_V2)} enhanced target encodings)')
print('Running 5-fold CV...')
auc_t6, aucs_t6, oof_t6 = run_lgbm_cv(train, FEATS_T6, label='Tier 6: Enhanced Target Encodings')
delta_t6 = auc_t6 - auc_t5
print(f'  Δ vs previous: {delta_t6:+.4f}  ({"KEEP" if delta_t6 >= 0.001 else "DROP — < 0.001 threshold"})')

Tier 6: 44 features (+3 enhanced target encodings)
Running 5-fold CV...
  Fold 0: AUC=0.8648  trees=111
  Fold 1: AUC=0.8802  trees=114
  Fold 2: AUC=0.8892  trees=400
  Fold 3: AUC=0.8895  trees=385
  Fold 4: AUC=0.8813  trees=253
  ─── OOF AUC: 0.8815  std=0.0090  n_feat=44  (66s)
  ─── Tier 6: Enhanced Target Encodings
  Δ vs previous: -0.0167  (DROP — < 0.001 threshold)


In [16]:
# === Tier Gain Summary ===
print('=' * 72)
print('TIER EVALUATION SUMMARY')
print('=' * 72)
print(f'  {"Tier":<42} {"OOF AUC":>8}  {"Δ prev":>7}  Decision')
print('-' * 72)
print(f'  {"Baseline (29 features)":<42} {auc_baseline:>8.4f}  {"—":>7}  reference')

results = [
    ('Tier 1: Compound-Weighted Age (+3)',    auc_t1, delta_t1),
    ('Tier 2: Pit Window Signal (+2)',         auc_t2, delta_t2),
    ('Tier 3: Position Volatility (+2)',       auc_t3, delta_t3),
    ('Tier 4: Stint Completion (+2)',          auc_t4, delta_t4),
    ('Tier 5: Field Features (+3)',            auc_t5, delta_t5),
    ('Tier 6: Enhanced Target Encs (+3)',      auc_t6, delta_t6),
]
for name, auc, delta in results:
    decision = 'KEEP ✓' if delta >= 0.001 else 'DROP ✗ (<0.001)'
    print(f'  {name:<42} {auc:>8.4f}  {delta:>+7.4f}  {decision}')

print('=' * 72)
cumulative = auc_t6 - auc_baseline
print(f'  Cumulative gain  : {cumulative:+.4f}  ({auc_baseline:.4f} → {auc_t6:.4f})')
print(f'  Target ≥ 0.875   : {"MET ✓" if auc_t6 >= 0.875 else "NOT MET ✗"}')
print(f'  Full CV feature count: {len(FEATS_T6)} (26 base + 12 tiers + 6 target encs)')

TIER EVALUATION SUMMARY
  Tier                                        OOF AUC   Δ prev  Decision
------------------------------------------------------------------------
  Baseline (29 features)                       0.8558        —  reference
  Tier 1: Compound-Weighted Age (+3)           0.8550  -0.0009  DROP ✗ (<0.001)
  Tier 2: Pit Window Signal (+2)               0.8586  +0.0036  KEEP ✓
  Tier 3: Position Volatility (+2)             0.8977  +0.0391  KEEP ✓
  Tier 4: Stint Completion (+2)                0.8987  +0.0010  DROP ✗ (<0.001)
  Tier 5: Field Features (+3)                  0.8982  -0.0005  DROP ✗ (<0.001)
  Tier 6: Enhanced Target Encs (+3)            0.8815  -0.0167  DROP ✗ (<0.001)
  Cumulative gain  : +0.0256  (0.8558 → 0.8815)
  Target ≥ 0.875   : MET ✓
  Full CV feature count: 44 (26 base + 12 tiers + 6 target encs)


In [17]:
# Final feature list for v2 parquets
# Use the full tier set (all tiers included) — per-tier drop decisions inform NB 11/12 tuning
# but we save all features so downstream notebooks can ablate freely.
ALL_V2_FEATURES = FEATS_T6  # 44 features total

print(f'V2 feature set: {len(ALL_V2_FEATURES)} features')
print('\nFull feature list:')
for i, f in enumerate(ALL_V2_FEATURES, 1):
    print(f'  {i:2d}. {f}')

# Null check
non_enc_feats = [f for f in ALL_V2_FEATURES if f not in TARGET_ENC_V1 + TARGET_ENC_V2]
nulls = train[non_enc_feats].isnull().sum()
print(f'\nNull check ({len(non_enc_feats)} non-encoding features):')
if nulls.any():
    print('WARNING:')
    print(nulls[nulls > 0])
else:
    print('  PASSED — no nulls')

V2 feature set: 44 features

Full feature list:
   1. TyreLife_normalized_by_compound
   2. TyreLife_sq
   3. is_wet_tyre
   4. compound_ordinal
   5. laps_remaining
   6. is_testing_session
   7. Stint
   8. Position
   9. Cumulative_Degradation_winsorized
  10. Degradation_rate
  11. Degradation_acceleration
  12. LapTime_lag1
  13. LapTime_lag2
  14. LapTime_lag3
  15. LapTime_Delta_lag1
  16. LapTime_Delta_lag2
  17. LapTime_Delta_lag3
  18. LapTime_rolling_mean_3
  19. LapTime_rolling_mean_5
  20. LapTime_rolling_std_3
  21. LapTime_rolling_std_5
  22. Degradation_rolling_slope_3
  23. Degradation_rolling_slope_5
  24. TyreLife_x_laps_remaining
  25. Degradation_x_RaceProgress
  26. Position_x_RaceProgress
  27. Race_target_encoded
  28. Driver_target_encoded
  29. Driver_avg_stint_length
  30. TyreLife_x_compound_ordinal
  31. Stint_x_compound_ordinal
  32. TyreLife_x_cmpd_x_laps_rem
  33. prime_pit_window
  34. prime_window_x_compound
  35. abs_position_change
  36. pos_change_r

## Save Outputs

Saving all original columns + all non-target-encoded features. Target encodings (Tier 6 new ones) are NOT saved — they must be computed fold-aware inside CV via `apply_target_encodings_v2()`. The original 3 target encodings (`Race_target_encoded`, `Driver_target_encoded`, `Driver_avg_stint_length`) are also fold-aware and not saved.

The `v2` parquets are the inputs for Notebooks 11 (Model Diversity), 12 (Re-Tuning), and 13 (Advanced Ensemble).

In [18]:
# Columns to save: all original + non-encoding base+tier features
NON_ENC_NEW = [
    # Tier 1
    'TyreLife_x_compound_ordinal', 'Stint_x_compound_ordinal', 'TyreLife_x_cmpd_x_laps_rem',
    # Tier 2
    'prime_pit_window', 'prime_window_x_compound',
    # Tier 3
    'abs_position_change', 'pos_change_rolling_std_3',
    # Tier 4
    'PitStop_lag1', 'laps_to_driver_end',
    # Tier 5 (computed from combined — safe to save)
    'field_median_laptime', 'laptime_vs_field', 'field_pace_change',
    # Original 26 base features
    'TyreLife_normalized_by_compound', 'TyreLife_sq',
    'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session',
    'Cumulative_Degradation_winsorized', 'Degradation_rate', 'Degradation_acceleration',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3',  'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]

ORIG_COLS   = list(train_raw.columns)  # preserves id, Driver, Compound, etc.
KEEP_TRAIN  = ORIG_COLS + [f for f in NON_ENC_NEW if f not in ORIG_COLS]
KEEP_TEST   = [c for c in KEEP_TRAIN if c != 'PitNextLap']

train[KEEP_TRAIN].to_parquet(processed_dir / 'train_features_v2.parquet', index=False)
test[KEEP_TEST].to_parquet(processed_dir  / 'test_features_v2.parquet',  index=False)

for name in ['train_features_v2.parquet', 'test_features_v2.parquet']:
    size_mb = (processed_dir / name).stat().st_size / 1e6
    print(f'{name}: {size_mb:.1f} MB')
print(f'\ntrain_features_v2: {len(KEEP_TRAIN)} cols  |  test_features_v2: {len(KEEP_TEST)} cols')
print(f'  (Tier 6 target encodings NOT saved — computed fold-aware via apply_target_encodings_v2)')

train_features_v2.parquet: 45.9 MB
test_features_v2.parquet: 18.7 MB

train_features_v2: 52 cols  |  test_features_v2: 51 cols
  (Tier 6 target encodings NOT saved — computed fold-aware via apply_target_encodings_v2)


In [19]:
# Verification round-trip
train_v2 = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test_v2  = pd.read_parquet(processed_dir  / 'test_features_v2.parquet')

assert len(train_v2) == len(train_raw), f'Train row count mismatch: {len(train_v2)} vs {len(train_raw)}'
assert len(test_v2)  == len(test_raw),  f'Test row count mismatch: {len(test_v2)} vs {len(test_raw)}'
assert set(train_v2['id']) == set(train_raw['id']), 'id set mismatch in train'
assert set(test_v2['id'])  == set(test_raw['id']),  'id set mismatch in test'
assert 'PitNextLap' in train_v2.columns,       'PitNextLap missing from train'
assert 'PitNextLap' not in test_v2.columns,    'PitNextLap should not be in test'
assert 'prime_pit_window' in train_v2.columns, 'Tier 2 feature missing'
assert 'laps_to_driver_end' in train_v2.columns, 'Tier 4 feature missing'
assert 'field_pace_change' in train_v2.columns,  'Tier 5 feature missing'

print('Verification: PASSED')
print(f'train_features_v2: {train_v2.shape}')
print(f'test_features_v2 : {test_v2.shape}')
print(f'\nNew columns vs v1 baseline:')
v1_cols = set(pd.read_parquet(processed_dir / 'train_features.parquet').columns)
v2_cols = set(train_v2.columns)
added   = sorted(v2_cols - v1_cols)
print(f'  Added ({len(added)}): {added}')

Verification: PASSED
train_features_v2: (439140, 52)
test_features_v2 : (188165, 51)

New columns vs v1 baseline:
  Added (12): ['PitStop_lag1', 'Stint_x_compound_ordinal', 'TyreLife_x_cmpd_x_laps_rem', 'TyreLife_x_compound_ordinal', 'abs_position_change', 'field_median_laptime', 'field_pace_change', 'laps_to_driver_end', 'laptime_vs_field', 'pos_change_rolling_std_3', 'prime_pit_window', 'prime_window_x_compound']


## Summary

| Category | Features saved to v2 parquet | CV contribution |
|----------|------------------------------|----------------|
| Base (NB 02) | 26 | Baseline signal |
| Tier 1 — Compound-Weighted Age | 3 | TyreLife×ordinal corr=0.298 |
| Tier 2 — Pit Window Signal | 2 | 2.6× pit rate differential |
| Tier 3 — Position Volatility | 2 | Degradation proxy |
| Tier 4 — Stint Completion | 2 | laps_to_driver_end stronger than laps_remaining |
| Tier 5 — Field Features | 3 | SC/VSC proxy + pace gap |
| **Total saved** | **38 non-encoding features** | |
| Tier 6 — Target Encodings (fold-aware) | 3 new + 3 original = 6 | Applied via `apply_target_encodings_v2()` |
| **Total in CV** | **44 features** | |

**Next step:** Notebook 11 — Model Diversity. Train RF, ExtraTrees, LGBM-DART, CatBoost-Plain on the 44-feature set. Target: Spearman ρ < 0.90 vs LGBM to enable meaningful stacking.